# Lesson 03 — SQLAlchemy ORM Models

**Run this in Google Colab** — every student gets the same environment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mpalomera/learning-sql/blob/master/lessons/06-sqlalchemy-orm-migrations/03_sqlalchemy_models.ipynb)

---

## Step 1: Install dependencies

In [ ]:
!pip install sqlalchemy oracledb -q

## Step 2: Define ORM models

In [61]:
from sqlalchemy import (
    create_engine, Column, Integer, String,
    Text, ForeignKey, DateTime, func
)
from sqlalchemy.orm import declarative_base, relationship, Session

Base = declarative_base()

class Team(Base):
    __tablename__ = "teams"
    id          = Column(Integer, primary_key=True)
    name        = Column(String(50), nullable=False, unique=True)
    description = Column(String(200))
    created_at  = Column(DateTime, server_default=func.current_timestamp())
    users = relationship("User", back_populates="team")
      # NEW COLUMNS:
    priority = Column(String(20), default='medium')
    due_date = Column(DateTime)
    tags = Column(String(500))

    def __repr__(self):
        return f"<Team(id={self.id}, name='{self.name}')>"

class User(Base):
    __tablename__ = "users"
    id         = Column(Integer, primary_key=True)
    username   = Column(String(50), nullable=False, unique=True)
    email      = Column(String(100), nullable=False)
    full_name  = Column(String(100))
    team_id    = Column(Integer, ForeignKey("teams.id"))
    created_at = Column(DateTime, server_default=func.current_timestamp())
    team  = relationship("Team", back_populates="users")
    tasks = relationship("Task", back_populates="assignee")
    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"

class Task(Base):
    __tablename__ = "tasks"
    id           = Column(Integer, primary_key=True)
    title        = Column(String(200), nullable=False)
    description  = Column(String(1000))
    status       = Column(String(20), default="open")
    assigned_to  = Column(Integer, ForeignKey("users.id"))
    created_at   = Column(DateTime, server_default=func.current_timestamp())
    updated_at   = Column(DateTime, onupdate=func.current_timestamp())
    assignee = relationship("User", back_populates="tasks")
    def __repr__(self):
        return f"<Task(id={self.id}, title='{self.title}', status='{self.status}')>"

print("✅ Models defined: Team, User, Task")

✅ Models defined: Team, User, Task


## Step 3: Connect and query with ORM

In [ ]:
# ============================================================
# CONFIGURATION — replace with your FreeSQL credentials
# ============================================================
USERNAME = ""
PASSWORD = ""   # <-- REPLACE THIS
DSN = "tcps://db.freesql.com:2484/23ai_34ui2"

engine = create_engine(
    "oracle+oracledb://:@",
    connect_args={
        "user": USERNAME,
        "password": PASSWORD,
        "dsn": DSN
    }
)

with Session(engine) as session:
    print("🏢 Teams:")
    for team in session.query(Team).all():
        print(f"   {team}")
        for user in team.users:
            print(f"      -> {user.full_name} ({user.username})")

    print("\n📝 Tasks with assignees:")
    for task in session.query(Task).all():
        assignee = task.assignee.full_name if task.assignee else "Unassigned"
        print(f"   {task.title} → {assignee}")

print("\n✅ ORM models working! Relationships navigate automatically.")

🏢 Teams:
   <Team(id=1, name='Engineering')>
      -> Alice Smith (alice_dev)
      -> Bob Jones (bob_dev)
   <Team(id=2, name='Product')>
      -> Carol White (carol_pm)

📝 Tasks with assignees:
   Fix login bug → Alice Smith
   Design new dashboard → Carol White
   Update dependencies → Bob Jones

✅ ORM models working! Relationships navigate automatically.


# Lesson 03 — Alembic Migrations (Pure Python)


In [ ]:
!pip install sqlalchemy oracledb alembic -q

In [ ]:
import os
from alembic.config import Config
from alembic import command

# Create a minimal alembic.ini in memory
alembic_cfg = Config()
alembic_cfg.set_main_option('script_location', '/content/alembic')
alembic_cfg.set_main_option('sqlalchemy.url', 'oracle+oracledb://:@')

# Initialize the migration directory
!mkdir -p /content/alembic/versions

# Write a minimal env.py
env_py = '''
from logging.config import fileConfig
from sqlalchemy import engine_from_config
from alembic import context

# This is the Alembic Config object
config = context.config

# Add your model's MetaData object here for 'autogenerate' support
from __main__ import Base
target_metadata = Base.metadata

def run_migrations_offline():
    url = config.get_main_option('sqlalchemy.url')
    context.configure(url=url, target_metadata=target_metadata, literal_binds=True)
    with context.begin_transaction():
        context.run_migrations()

def run_migrations_online():
    connectable = engine_from_config(
        config.get_section(config.config_ini_section),
        prefix='sqlalchemy.',
        connect_args={'user': "''' + USERNAME + '''", 'password': "''' + PASSWORD + '''", 'dsn': "''' + DSN + '''"}
    )
    with connectable.connect() as connection:
        context.configure(connection=connection, target_metadata=target_metadata)
        with context.begin_transaction():
            context.run_migrations()

if context.is_offline_mode():
    run_migrations_offline()
else:
    run_migrations_online()
'''

with open('/content/alembic/env.py', 'w') as f:
    f.write(env_py)

# Write a minimal script.py.mako
script_template = '''"""${message}

Revision ID: ${up_revision}
Revises: ${down_revision | comma,n}
Create Date: ${create_date}
"""

from alembic import op
import sqlalchemy as sa
${imports if imports else ""}

# revision identifiers, used by Alembic.
revision = ${repr(up_revision)}
down_revision = ${repr(down_revision)}
branch_labels = ${repr(branch_labels)}
depends_on = ${repr(depends_on)}

def upgrade():
${upgrades if upgrades else "    pass"}

def downgrade():
${downgrades if downgrades else "    pass"}
'''

with open('/content/alembic/script.py.mako', 'w') as f:
    f.write(script_template)

print('✅ Alembic initialized in /content/alembic/')

✅ Alembic initialized in /content/alembic/


In [62]:
# Generate migration from models vs current database state
command.revision(alembic_cfg, autogenerate=True, message='Initial schema')

# Show what was generated
import glob
migration_files = sorted(glob.glob('/content/alembic/versions/*.py'))
print('Generated migrations:')
for f in migration_files:
    print(f'  {f}')

Generating /content/alembic/versions/dc62f0e11bfa_initial_schema.py ...  done
Generated migrations:
  /content/alembic/versions/d67fc4608479_initial_schema.py
  /content/alembic/versions/dc62f0e11bfa_initial_schema.py


In [64]:
# Read and display the latest migration
latest = migration_files[1]
with open(latest) as f:
    content = f.read()

print(content)


"""Initial schema

Revision ID: dc62f0e11bfa
Revises: d67fc4608479
Create Date: 2026-05-12 13:49:07.882812
"""

from alembic import op
import sqlalchemy as sa


# revision identifiers, used by Alembic.
revision = 'dc62f0e11bfa'
down_revision = 'd67fc4608479'
branch_labels = None
depends_on = None

def upgrade():
# ### commands auto generated by Alembic - please adjust! ###
    op.add_column('teams', sa.Column('priority', sa.String(length=20), nullable=True))
    op.add_column('teams', sa.Column('due_date', sa.DateTime(), nullable=True))
    op.add_column('teams', sa.Column('tags', sa.String(length=500), nullable=True))
    # ### end Alembic commands ###

def downgrade():
# ### commands auto generated by Alembic - please adjust! ###
    op.drop_column('teams', 'tags')
    op.drop_column('teams', 'due_date')
    op.drop_column('teams', 'priority')
    # ### end Alembic commands ###



In [ ]:
team = Team()

In [65]:

command.upgrade(alembic_cfg, 'head')
print('✅ Migration applied! Tables created.')

✅ Migration applied! Tables created.


In [66]:
# Rollback one migration
command.downgrade(alembic_cfg, '-1')
print('✅ Downgraded by 1. New columns removed.')

✅ Downgraded by 1. New columns removed.


In [ ]:
import glob
import os

# Delete all generated migration files
migration_files = glob.glob('/content/alembic/versions/*.py')

for f in migration_files:
    os.remove(f)
    print(f"Deleted: {f}")

---
# Exercise 1 — Comment Model

**Questions answered:**
1. `Comment` should have relationships to both `Task` (many-to-one) and `User` (many-to-one).
2. Yes — `Task` should have a `comments` back-reference so you can do `task.comments`.
3. When a task is deleted its comments should also be deleted (`cascade='all, delete-orphan'` on the ORM side; `ON DELETE CASCADE` on the FK).

In [ ]:
from sqlalchemy import (
    create_engine, Column, Integer, String,
    Text, ForeignKey, DateTime, func, CheckConstraint
)
from sqlalchemy.orm import declarative_base, relationship, Session

Base = declarative_base()

class Team(Base):
    __tablename__ = 'teams'
    id          = Column(Integer, primary_key=True)
    name        = Column(String(50), nullable=False, unique=True)
    description = Column(String(200))
    created_at  = Column(DateTime, server_default=func.current_timestamp())
    users = relationship('User', back_populates='team')

    def __repr__(self):
        return f"<Team(id={self.id}, name='{self.name}')>"

class User(Base):
    __tablename__ = 'users'
    id         = Column(Integer, primary_key=True)
    username   = Column(String(50), nullable=False, unique=True)
    email      = Column(String(100), nullable=False)
    full_name  = Column(String(100))
    team_id    = Column(Integer, ForeignKey('teams.id'))
    created_at = Column(DateTime, server_default=func.current_timestamp())
    team     = relationship('Team', back_populates='users')
    tasks    = relationship('Task', back_populates='assignee')
    comments = relationship('Comment', back_populates='author')

    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"

class Task(Base):
    __tablename__ = 'tasks'
    id          = Column(Integer, primary_key=True)
    title       = Column(String(200), nullable=False)
    description = Column(String(1000))
    status      = Column(String(20), default='open')
    assigned_to = Column(Integer, ForeignKey('users.id'))
    created_at  = Column(DateTime, server_default=func.current_timestamp())
    updated_at  = Column(DateTime, onupdate=func.current_timestamp())
    assignee = relationship('User', back_populates='tasks')
    # A task owns its comments; deleting the task cascades to comments
    comments = relationship('Comment', back_populates='task',
                             cascade='all, delete-orphan')

    def __repr__(self):
        return f"<Task(id={self.id}, title='{self.title}', status='{self.status}')>"

class Comment(Base):
    __tablename__ = 'comments'
    __table_args__ = (CheckConstraint("content != ''", name='ck_comments_content'),)

    id         = Column(Integer, primary_key=True)
    task_id    = Column(Integer, ForeignKey('tasks.id', ondelete='CASCADE'), nullable=False)
    user_id    = Column(Integer, ForeignKey('users.id'), nullable=False)
    content    = Column(Text, nullable=False)
    created_at = Column(DateTime, server_default=func.current_timestamp())

    task   = relationship('Task', back_populates='comments')
    author = relationship('User', back_populates='comments')

    def __repr__(self):
        return f"<Comment(id={self.id}, task_id={self.task_id}, user_id={self.user_id})>"

print('✅ Comment model defined with Task and User relationships')
print('   - task.comments  → all comments on a task')
print('   - comment.author → the user who wrote it')
print('   - cascade delete-orphan: deleting a task removes its comments')

---
# Exercise 2 — Migration Creation

**Questions answered:**
1. `upgrade()` runs `op.create_table('comments', ...)` — it creates the new table in the database.
2. `downgrade()` runs `op.drop_table('comments')` — it removes the table, reverting the change.
3. If you downgrade this migration all rows in `comments` are permanently deleted and the table is dropped.

**Bonus:** The `CheckConstraint("content != ''")` is declared on the model's `__table_args__` (already included in Exercise 1 above).

In [ ]:
import os, glob
from alembic.config import Config
from alembic import command

# Re-use the alembic config created earlier in the lesson
alembic_cfg = Config()
alembic_cfg.set_main_option('script_location', '/content/alembic')
alembic_cfg.set_main_option('sqlalchemy.url', 'oracle+oracledb://:@')

# Generate migration from current model state
command.revision(
    alembic_cfg,
    autogenerate=True,
    message='add comments table'
)

# List generated migration files
migration_files = sorted(glob.glob('/content/alembic/versions/*.py'))
print('Generated migration files:')
for f in migration_files:
    print(f'  {f}')

In [ ]:
# Inspect the latest migration
latest = migration_files[-1]
with open(latest) as f:
    print(f.read())

---
# Exercise 3 — CRUD Challenge

In [ ]:
# Priorities: high > medium > low
PRIORITY = {'high': 1, 'medium': 2, 'low': 3}

with Session(engine) as session:
    # 1. Create DevOps team
    devops = Team(name='DevOps', description='DevOps and infrastructure team')
    session.add(devops)
    session.flush()  # get devops.id before creating the user

    # 2. Create diana_ops linked to that team
    diana = User(
        username='diana_ops',
        email='diana@example.com',
        full_name='Diana Ops',
        team_id=devops.id
    )
    session.add(diana)
    session.flush()

    # 3. Create 3 tasks with different priorities
    t_high = Task(
        title='Set up CI/CD pipeline',
        description='Configure GitHub Actions for automated deployments',
        status='open',
        assigned_to=diana.id
    )
    t_medium = Task(
        title='Monitor server health',
        description='Install and configure Prometheus + Grafana',
        status='open',
        assigned_to=diana.id
    )
    t_low = Task(
        title='Update SSL certificates',
        description='Renew expiring certs on staging servers',
        status='open',
        assigned_to=diana.id
    )
    session.add_all([t_high, t_medium, t_low])
    session.flush()

    # 4. Print task count
    total = session.query(Task).count()
    print(f'📊 Total tasks in DB: {total}')

    # 5. Close the highest-priority task
    t_high.status = 'closed'
    print(f'✅ Closed task: "{t_high.title}"')

    # 6. Delete the lowest-priority task
    session.delete(t_low)
    print(f'🗑️  Deleted task: "{t_low.title}"')

    session.commit()

    # Show remaining DevOps tasks via relationship
    session.refresh(diana)
    print(f'\n📝 Remaining tasks for {diana.full_name}:')
    for t in diana.tasks:
        print(f'   {t}')

---
# Exercise 4 — Migration Rollback

**Questions answered:**
1. After `downgrade(-1)` the `estimated_hours` column is dropped from the `tasks` table — it no longer exists in the schema.
2. Any data that was stored in `estimated_hours` is permanently lost; Oracle drops the column and its values with no automatic backup.

In [ ]:
import os, glob
from alembic.config import Config
from alembic import command

alembic_cfg = Config()
alembic_cfg.set_main_option('script_location', '/content/alembic')
alembic_cfg.set_main_option('sqlalchemy.url', 'oracle+oracledb://:@')

# Simulate: manually write a migration that adds estimated_hours
bad_migration = '''
"""add estimated_hours

Revision ID: bad00000
Revises:
Create Date: 2026-05-12
"""
from alembic import op
import sqlalchemy as sa

revision = 'bad00000'
down_revision = None
branch_labels = None
depends_on = None

def upgrade():
    op.add_column('tasks', sa.Column('estimated_hours', sa.Integer(), nullable=True))

def downgrade():
    op.drop_column('tasks', 'estimated_hours')
'''

with open('/content/alembic/versions/bad00000_add_estimated_hours.py', 'w') as f:
    f.write(bad_migration)

# Apply the bad migration
command.upgrade(alembic_cfg, 'head')
print('⚠️  Bad migration applied — estimated_hours column now exists')

# Roll it back
command.downgrade(alembic_cfg, '-1')
print('✅ Downgraded by 1 — estimated_hours column removed')
print('   Any data in that column is gone permanently.')

---
# Exercise 5 — Concept Check

1. **Why use ORM instead of raw SQL?**  
   ORM maps database rows to Python objects, so you work with classes and attributes instead of hand-written SQL strings. It reduces boilerplate, catches errors at the Python level, and makes relationships (e.g. `task.assignee`) feel natural. You can also switch databases by changing the connection string without rewriting queries.

2. **Why use migrations?**  
   Migrations version-control schema changes the same way Git versions code. Every change is reproducible, reviewable, and reversible. Without migrations you would need to manually ALTER tables on every environment and keep them in sync.

3. **When would you rollback?**  
   When a deployed migration causes a bug, breaks an application, or introduces bad data — you rollback to return the schema to the last known-good state while you fix the migration.

4. **Difference between `add()` and `commit()`?**  
   `session.add(obj)` stages an object in the unit-of-work (SQLAlchemy tracks it but no SQL is sent yet). `session.commit()` flushes all pending changes to the database and ends the transaction — only then are the rows actually written and durable.

5. **Why are relationships useful?**  
   They let you navigate between related objects with Python attribute access (`task.assignee`, `team.users`) instead of writing JOIN queries. SQLAlchemy handles the JOIN or sub-query automatically, and `cascade` options mean you don't have to manually manage dependent rows.